# Complexity metrics analysis for significant results


**Source file:** `all_group_differences.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 120)

CSV_PATH = '../all_group_differences.csv'
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(1188, 8)


,measure,p_omnibus,epsilon_squared,comparison,dunn_p_holm,cliffs_delta,higher_group,pair_flag
0,normalized_helpfulness_responses_qwen-3.5-27B,0.112036,0.005989,low vs middle,0.332234,-0.052208,middle,omnibus not significant
1,normalized_helpfulness_responses_qwen-3.5-27B,0.112036,0.005989,low vs upper,0.110884,-0.151327,upper,omnibus not significant
2,normalized_helpfulness_responses_qwen-3.5-27B,0.112036,0.005989,middle vs upper,0.332234,-0.079719,upper,omnibus not significant
3,normalized_helpfulness_responses_llama-3.3-70B,0.031422,0.009467,low vs middle,0.211209,-0.073316,middle,no significant difference (pair)
4,normalized_helpfulness_responses_llama-3.3-70B,0.031422,0.009467,low vs upper,0.027489,-0.184579,upper,significant difference (pair)


## Select `measure` into `model` + `metric`, keep only `cplx_*`



In [4]:
MODELS = ['qwen-3.5-27B', 'llama-3.3-70B', 'gemma-4-31B', 'gpt-5.5']

def get_model(measure: str) -> str:
    for m in MODELS:
        if measure.endswith(m):
            return m
    raise ValueError(f'Unrecognized model suffix in: {measure}')

df['model'] = df['measure'].apply(get_model)
df['metric'] = df.apply(lambda r: r['measure'][: -(len(r['model']) + 1)], axis=1)

cplx = df[df['metric'].str.startswith('cplx_')].copy()
cplx['omnibus_significant'] = cplx['p_omnibus'] < 0.05
cplx['pairwise_significant'] = cplx['pair_flag'] == 'significant difference (pair)'

print(f"cplx_* rows: {len(cplx)}  |  unique cplx metrics: {cplx['metric'].nunique()}  |  models: {cplx['model'].nunique()}")
cplx.head()

cplx_* rows: 936  |  unique cplx metrics: 78  |  models: 4


,measure,p_omnibus,epsilon_squared,comparison,dunn_p_holm,cliffs_delta,higher_group,pair_flag,model,metric,omnibus_significant,pairwise_significant
72,cplx_additive_connectives_gemma-4-31B,0.924105,0.000216,low vs middle,1.000000,0.008586,low,omnibus not significant,gemma-4-31B,cplx_additive_connectives,False,False
73,cplx_additive_connectives_gemma-4-31B,0.924105,0.000216,low vs upper,1.000000,-0.031500,upper,omnibus not significant,gemma-4-31B,cplx_additive_connectives,False,False
74,cplx_additive_connectives_gemma-4-31B,0.924105,0.000216,middle vs upper,1.000000,-0.018055,upper,omnibus not significant,gemma-4-31B,cplx_additive_connectives,False,False
75,cplx_adjectives_density_gemma-4-31B,0.287807,0.003408,low vs middle,0.344305,0.073707,low,omnibus not significant,gemma-4-31B,cplx_adjectives_density,False,False
76,cplx_adjectives_density_gemma-4-31B,0.287807,0.003408,low vs upper,0.974051,0.058662,low,omnibus not significant,gemma-4-31B,cplx_adjectives_density,False,False


In [5]:
# One row per (model, metric) for omnibus-level facts
omni = cplx.drop_duplicates(subset=['model', 'metric'])[
    ['model', 'metric', 'p_omnibus', 'epsilon_squared', 'omnibus_significant']
].reset_index(drop=True)

# Count of significant pairwise comparisons per (model, metric)
pair_sig_counts = (
    cplx.groupby(['model', 'metric'])['pairwise_significant'].sum().rename('n_sig_pairs')
)
omni = omni.merge(pair_sig_counts, on=['model', 'metric'])

summary_per_model = omni.groupby('model').agg(
    n_metrics=('metric', 'nunique'),
    n_omnibus_significant=('omnibus_significant', 'sum'),
    n_metrics_with_any_sig_pair=('n_sig_pairs', lambda s: (s > 0).sum()),
    n_metrics_all_3_pairs_sig=('n_sig_pairs', lambda s: (s == 3).sum()),
).sort_values('n_omnibus_significant', ascending=False)
summary_per_model['pct_omnibus_significant'] = (
    100 * summary_per_model['n_omnibus_significant'] / summary_per_model['n_metrics']
).round(1)
summary_per_model

,n_metrics,n_omnibus_significant,n_metrics_with_any_sig_pair,n_metrics_all_3_pairs_sig,pct_omnibus_significant
model,,,,,
gpt-5.5,78,33,30,1,42.3
llama-3.3-70B,78,21,20,4,26.9
qwen-3.5-27B,78,19,16,3,24.4
gemma-4-31B,78,13,10,0,16.7


## Metrics significant across ALL models


In [9]:
pivot_p = omni.pivot(index='metric', columns='model', values='p_omnibus')[MODELS]
pivot_sig = omni.pivot(index='metric', columns='model', values='omnibus_significant')[MODELS]

all4_metrics = pivot_sig.index[pivot_sig.all(axis=1)].tolist()
print(f"{len(all4_metrics)} metrics are omnibus-significant in ALL 4 models:\n")
for m in all4_metrics:
    print(' -', m)

core = omni[omni['metric'].isin(all4_metrics)].pivot(index='metric', columns='model', values='epsilon_squared')[MODELS]
core.round(4)

7 metrics are omnibus-significant in ALL 4 models:

 - cplx_adversative_connectives
 - cplx_average_age_of_acquisition
 - cplx_first_person_pronouns_density
 - cplx_median_kuperman_age_of_acquisition
 - cplx_negations_density
 - cplx_syllables_per_word
 - cplx_third_person_pronouns_density


model,qwen-3.5-27B,llama-3.3-70B,gemma-4-31B,gpt-5.5
metric,,,,
cplx_adversative_connectives,0.0095,0.0117,0.0105,0.0197
cplx_average_age_of_acquisition,0.0275,0.0224,0.0152,0.0185
cplx_first_person_pronouns_density,0.0120,0.0169,0.0165,0.0177
cplx_median_kuperman_age_of_acquisition,0.0310,0.0235,0.0170,0.0151
cplx_negations_density,0.0136,0.0199,0.0294,0.0462
cplx_syllables_per_word,0.0235,0.0230,0.0173,0.0250
cplx_third_person_pronouns_density,0.0097,0.0160,0.0084,0.0099


### Direction of the effect for the universal metrics


In [10]:
direction = cplx[cplx['metric'].isin(all4_metrics)].copy()
direction_tbl = direction.pivot_table(
    index='metric', columns=['comparison'], values='higher_group', aggfunc=lambda s: '/'.join(s.unique())
)
direction_tbl = direction_tbl.loc[all4_metrics]
direction_tbl

comparison,low vs middle,low vs upper,middle vs upper
metric,,,
cplx_adversative_connectives,low,low/upper,upper
cplx_average_age_of_acquisition,middle,upper,upper
cplx_first_person_pronouns_density,low,upper/low,upper
cplx_median_kuperman_age_of_acquisition,middle,upper,upper
cplx_negations_density,low,upper/low,upper
cplx_syllables_per_word,middle,upper,upper
cplx_third_person_pronouns_density,low,upper/low,upper
